In [ ]:
#The Problem Before the Syntax

In [15]:
import pandas as pd
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 70)
 
toy_sales = pd.DataFrame({
    "stall": ["S1", "S1", "S2", "S2", "S3", "S1", "S2", "S3", "S4"],
    "month": ["2026-01", "2026-01", "2026-01", "2026-01", "2026-01",
              "2026-02", "2026-02", "2026-02", "2026-02"],
    "item":  ["Chai", "Coffee", "Chai", "Chai", "Coffee",
              "Chai", "Coffee", "Chai", "Chai"],
    "cups":  [120, 40, 90, 30, 60, 150, 45, 80, 25],
})
toy_stalls = pd.DataFrame({
    "stall": ["S1", "S2", "S3", "S5"],
    "city":  ["Pune", "Pune", "Indore", "Bhopal"],
})
print(toy_sales)
print()
print(toy_stalls)


  stall    month    item  cups
0    S1  2026-01    Chai   120
1    S1  2026-01  Coffee    40
2    S2  2026-01    Chai    90
3    S2  2026-01    Chai    30
4    S3  2026-01  Coffee    60
5    S1  2026-02    Chai   150
6    S2  2026-02  Coffee    45
7    S3  2026-02    Chai    80
8    S4  2026-02    Chai    25

  stall    city
0    S1    Pune
1    S2    Pune
2    S3  Indore
3    S5  Bhopal


In [ ]:
#1. Split, Apply, Combine: groupby()1. Split, Apply, Combine: groupby()

In [16]:
cups_per_stall = toy_sales.groupby("stall")["cups"].sum()
print(cups_per_stall)

stall
S1    310
S2    165
S3    140
S4     25
Name: cups, dtype: int64


In [17]:
print(toy_sales.groupby(["stall", "month"])["cups"].sum())


stall  month  
S1     2026-01    160
       2026-02    150
S2     2026-01    120
       2026-02     45
S3     2026-01     60
       2026-02     80
S4     2026-02     25
Name: cups, dtype: int64


In [ ]:
#The same idea on the real data

In [18]:
import os
 
def find_repo_root(marker_folder="data", start=None):
    """Search upward from the current folder for the folder that contains
    `data` -- that folder is the repo root."""
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(current, marker_folder)):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            raise FileNotFoundError(f"No '{marker_folder}' folder found")
        current = parent
 
REPO_ROOT = find_repo_root()
RAW = os.path.join(REPO_ROOT, "data", "raw")
TEXT_COLUMNS = {"safety_report_id": str, "serious": str,
                "patient_age_unit": str, "country": str, "event_date": str}
 
EVENTS_FILE = os.path.join(RAW, "adverse_events_10k.csv")
df = pd.read_csv(EVENTS_FILE, dtype=TEXT_COLUMNS)
print(df.shape)


(6218, 8)


In [19]:
reports_per_drug = df.groupby("drug_name")["safety_report_id"].count()
print("groups:", len(reports_per_drug))
print(reports_per_drug.sort_values(ascending=False).head(5))
print()
print("different reactions reported per drug, top 3:")
variety = df.groupby("drug_name")["reaction"].nunique()
print(variety.sort_values(ascending=False).head(3))

groups: 1125
drug_name
DUPIXENT     621
CABOMETYX    170
SKYRIZI      139
RINVOQ       105
RYALTRIS      79
Name: safety_report_id, dtype: int64

different reactions reported per drug, top 3:
drug_name
DUPIXENT     223
SKYRIZI      106
CABOMETYX    103
Name: reaction, dtype: int64


In [ ]:
#2. Many Answers at Once: agg()

In [20]:
per_stall = toy_sales.groupby("stall").agg(
    sales=("cups", "count"),
    total_cups=("cups", "sum"),
    biggest_sale=("cups", "max"),
)
print(per_stall)

       sales  total_cups  biggest_sale
stall                                 
S1         3         310           150
S2         3         165            90
S3         2         140            80
S4         1          25            25


In [ ]:
#Turning a yes/no code into a share

In [21]:
df["is_serious"] = df["serious"].map({"1": 1.0, "2": 0.0})
print(df["is_serious"].value_counts(dropna=False))

is_serious
0.0    3288
1.0    2917
NaN      13
Name: count, dtype: int64


In [22]:
by_country = df.groupby("country").agg(
    reports=("safety_report_id", "count"),
    drugs=("drug_name", "nunique"),
    serious_share=("is_serious", "mean"),
)
big = by_country[by_country["reports"] >= 100]
big = big.sort_values("reports", ascending=False)
print(big.round(3))


         reports  drugs  serious_share
country                               
US          4234    694          0.293
GB           253    119          0.976
EU           229    148          0.921
JP           186    117          0.866
CA           170     79          0.947
CN           129     93          0.992


In [ ]:
#size() versus count(), and as_index=False

In [23]:
by_drug = df.groupby("drug_name")
print("size():           ", by_drug.size().sum())
print("count() of serious:", by_drug["serious"].count().sum())

size():            6218
count() of serious: 6205


In [24]:
flat = toy_sales.groupby(["stall", "item"], as_index=False)
print(flat.agg(total_cups=("cups", "sum")))

  stall    item  total_cups
0    S1    Chai         270
1    S1  Coffee          40
2    S2    Chai         120
3    S2  Coffee          45
4    S3    Chai          80
5    S3  Coffee          60
6    S4    Chai          25


In [ ]:
#3. Joining Tables: merge()

In [25]:
for how in ["inner", "left", "right", "outer"]:
    joined = toy_sales.merge(toy_stalls, on="stall", how=how)
    print(how, "->", len(joined), "rows")
 
print()
print(toy_sales.merge(toy_stalls, on="stall", how="outer").tail(3))

inner -> 8 rows
left -> 9 rows
right -> 9 rows
outer -> 10 rows

  stall    month  item  cups    city
7    S3  2026-02  Chai  80.0  Indore
8    S4  2026-02  Chai  25.0     NaN
9    S5      NaN   NaN   NaN  Bhopal


In [ ]:
#Bringing in the drug class

In [26]:
drug_ref = pd.read_csv(os.path.join(RAW, "drug_reference.csv"))
print(drug_ref.shape)
print(drug_ref.head(3))
print()
print("one row per drug?", drug_ref["drug_name"].is_unique)


(521, 3)
       drug_name                          drug_class  n_classes
0       ABRAXANE               Microtubule Inhibitor          1
1         ABREVA  Non-Standardized Chemical Allergen          1
2  ACALABRUTINIB                    Kinase Inhibitor          1

one row per drug? True


In [27]:
with_class = df.merge(drug_ref, on="drug_name", how="left",
                      validate="many_to_one", indicator=True)
print("rows before:", len(df), "| rows after:", len(with_class))
print(with_class["_merge"].value_counts())

rows before: 6218 | rows after: 6218
_merge
both          3987
left_only     2231
right_only       0
Name: count, dtype: int64


In [ ]:
#Drugs with more than one class

In [28]:
multi_drugs = drug_ref[drug_ref["n_classes"] > 1]
print("drugs with more than one class:", len(multi_drugs))
in_multi = with_class["drug_name"].isin(multi_drugs["drug_name"])
print("their reports:", in_multi.sum())
examples = ["BOTOX", "BREZTRI", "STELARA", "ZEPBOUND"]
print(multi_drugs[multi_drugs["drug_name"].isin(examples)])

drugs with more than one class: 50
their reports: 418
    drug_name                                                     drug_class  n_classes
57      BOTOX                                Acetylcholine Release Inhibitor          2
61    BREZTRI                                       beta2-Adrenergic Agonist          2
425   STELARA                                      Interleukin-12 Antagonist          2
513  ZEPBOUND  Glucose-dependent Insulinotropic Polypeptide Receptor Agonist          2


In [ ]:
#4. Choosing the Right Join Is a Business Decision

In [29]:
inner = df.merge(drug_ref, on="drug_name")
left = df.merge(drug_ref, on="drug_name", how="left")
print("inner:", len(inner), "reports")
print("left: ", len(left), "reports")

inner: 3987 reports
left:  6218 reports


In [30]:
no_class = with_class[with_class["_merge"] == "left_only"]
print(no_class["drug_name"].value_counts().head(5))

drug_name
CABOMETYX    170
RYALTRIS      79
HUMIRA        71
CREON         55
NUPLAZID      55
Name: count, dtype: int64


In [31]:
with_class["drug_class"] = with_class["drug_class"].fillna("Unclassified")
print(with_class["drug_class"].value_counts().head(3))

drug_class
Unclassified                               2231
Interleukin-4 Receptor alpha Antagonist     630
Interleukin-23 Antagonist                   171
Name: count, dtype: int64


In [ ]:
#Why validate= matters

In [32]:
dupixent_row = drug_ref[drug_ref["drug_name"] == "DUPIXENT"]
broken_ref = pd.concat([drug_ref, dupixent_row])
broken = df.merge(broken_ref, on="drug_name", how="left")
print("rows after merging with the broken table:", len(broken))
try:
    df.merge(broken_ref, on="drug_name", how="left", validate="many_to_one")
except pd.errors.MergeError as err:
    print("with validate:", str(err).splitlines()[0])

rows after merging with the broken table: 6839
with validate: Merge keys are not unique in right dataset; not a many-to-one merge


In [ ]:
# 5. Grids: pivot_table() and crosstab()

In [34]:
grid = toy_sales.pivot_table(index="stall", columns="month", values="cups",
                             aggfunc="sum", fill_value=0)
print(grid)

month  2026-01  2026-02
stall                  
S1         160      150
S2         120       45
S3          60       80
S4           0       25


In [35]:
print(pd.crosstab(toy_sales["stall"], toy_sales["item"]))
print()
shares = pd.crosstab(toy_sales["stall"], toy_sales["item"], normalize="index")
print(shares.round(2))


item   Chai  Coffee
stall              
S1        2       1
S2        2       1
S3        1       1
S4        1       0

item   Chai  Coffee
stall              
S1     0.67    0.33
S2     0.67    0.33
S3     0.50    0.50
S4     1.00    0.00


In [ ]:
# Bringing in the reaction category

In [36]:
reaction_map = pd.read_csv(os.path.join(RAW, "reaction_categories.csv"))
rx = with_class.drop(columns="_merge")
rx = rx.merge(reaction_map, on="reaction", how="left", validate="many_to_one")
print("reports with a category:", rx["reaction_category"].notna().sum())
print("reports without one:    ", rx["reaction_category"].isna().sum())

reports with a category: 4283
reports without one:     1935


In [ ]:
# The trap: grids leave out missing labels

In [37]:
too_early = pd.crosstab(rx["reaction_category"], rx["serious"])
print("reports in the grid:", too_early.values.sum())

reports in the grid: 4274


In [38]:
rx["reaction_category"] = rx["reaction_category"].fillna("Unmapped")
rx["region"] = "Outside US"
rx.loc[rx["country"] == "US", "region"] = "US"
rx.loc[rx["country"].isna(), "region"] = "Not reported"
 
counts = pd.crosstab(rx["reaction_category"], rx["region"])
mix = pd.crosstab(rx["reaction_category"], rx["region"], normalize="columns")
print(mix.round(3))
print()
print("reports counted:", counts.values.sum())


region                            Not reported  Outside US     US
reaction_category                                                
Brain, nerves and mood                   0.069       0.058  0.062
Death or hospital care                   0.112       0.085  0.054
General symptoms                         0.055       0.033  0.064
Gut and liver                            0.035       0.051  0.054
Heart, blood and circulation             0.062       0.073  0.043
Infection and breathing                  0.082       0.046  0.052
Lack of effect                           0.097       0.032  0.052
Muscles, bones and joints                0.042       0.034  0.034
Other organs and cancer                  0.030       0.062  0.051
Product use or dosing error              0.037       0.043  0.168
Skin, allergy and injection site         0.094       0.034  0.104
Unmapped                                 0.285       0.450  0.262

reports counted: 6218


In [ ]:
# pivot_table versus crosstab

In [39]:
totals = rx.pivot_table(index="region", columns="serious",
                        values="safety_report_id", aggfunc="count",
                        fill_value=0, margins=True)
print(totals)

serious          1     2   All
region                        
Not reported   229   174   403
Outside US    1450   130  1580
US            1238  2984  4222
All           2917  3288  6205


In [ ]:
# 6. Long and Wide: stack() and unstack()

In [40]:
long_form = grid.stack()
print(long_form)
print()
print("back to wide:")
print(long_form.unstack())

stall  month  
S1     2026-01    160
       2026-02    150
S2     2026-01    120
       2026-02     45
S3     2026-01     60
       2026-02     80
S4     2026-01      0
       2026-02     25
dtype: int64

back to wide:
month  2026-01  2026-02
stall                  
S1         160      150
S2         120       45
S3          60       80
S4           0       25


In [41]:
monthly = toy_sales.groupby(["month", "stall"])["cups"].sum()
print(monthly.unstack("stall", fill_value=0))

stall     S1   S2  S3  S4
month                    
2026-01  160  120  60   0
2026-02  150   45  80  25


In [ ]:
# 7. Change Over Time: cumsum() and pct_change()

In [42]:
print("running total of cups, per stall:")
print(grid.T.cumsum())
print()
print("change from January to February:")
print(grid.T.pct_change().round(2))

running total of cups, per stall:
stall     S1   S2   S3  S4
month                     
2026-01  160  120   60   0
2026-02  310  165  140  25

change from January to February:
stall      S1    S2    S3   S4
month                         
2026-01   NaN   NaN   NaN  NaN
2026-02 -0.06 -0.62  0.33  inf


In [43]:
import numpy as np
print(grid.T.pct_change().replace([np.inf, -np.inf], np.nan).round(2))

stall      S1    S2    S3  S4
month                        
2026-01   NaN   NaN   NaN NaN
2026-02 -0.06 -0.62  0.33 NaN


In [ ]:
# Quarters on the real data

In [44]:
dates = pd.to_datetime(df["event_date"], format="%Y%m%d")
df["quarter"] = dates.dt.to_period("Q").astype(str)
serious_by_q = df.groupby("quarter").agg(
    serious=("is_serious", "sum"),
    reports=("safety_report_id", "count"),
)
serious_by_q["serious"] = serious_by_q["serious"].astype(int)
serious_by_q["change"] = serious_by_q["serious"].pct_change().round(3)
serious_by_q["running_total"] = serious_by_q["serious"].cumsum()
print(serious_by_q)

         serious  reports  change  running_total
quarter                                         
2024Q3       376      752     NaN            376
2024Q4       409      741   0.088            785
2025Q1       320      797  -0.218           1105
2025Q2       434      774   0.356           1539
2025Q3       351      730  -0.191           1890
2025Q4       352      769   0.003           2242
2026Q1       248      833  -0.295           2490
2026Q2       427      822   0.722           2917


In [ ]:
# The trap: pct_change() does not know where a group ends

In [45]:
rx["quarter"] = df["quarter"]   # same rows, same order: rx came from df
by_region = rx.groupby(["region", "quarter"]).size()
print(by_region.pct_change().round(3).head(10))

region        quarter
Not reported  2024Q3       NaN
              2024Q4    -0.373
              2025Q1     0.062
              2025Q2     1.441
              2025Q3    -0.675
              2025Q4     1.481
              2026Q1    -0.642
              2026Q2     2.542
Outside US    2024Q3     1.576
              2024Q4    -0.027
dtype: float64


In [46]:
print("fix 1, grouped pct_change:")
grouped = by_region.groupby(level="region").pct_change()
print(grouped.loc["Outside US"].round(3).head(3))
print()
print("fix 2, one column per group:")
wide = by_region.unstack("region", fill_value=0)
print(wide.pct_change().round(3).head(3))
print("fix 1, grouped pct_change:")
grouped = by_region.groupby(level="region").pct_change()
print(grouped.loc["Outside US"].round(3).head(3))
print()
print("fix 2, one column per group:")
wide = by_region.unstack("region", fill_value=0)
print(wide.pct_change().round(3).head(3))

fix 1, grouped pct_change:
quarter
2024Q3      NaN
2024Q4   -0.027
2025Q1   -0.080
dtype: float64

fix 2, one column per group:
region   Not reported  Outside US     US
quarter                                 
2024Q3            NaN         NaN    NaN
2024Q4         -0.373      -0.027  0.029
2025Q1          0.062      -0.080  0.143
fix 1, grouped pct_change:
quarter
2024Q3      NaN
2024Q4   -0.027
2025Q1   -0.080
dtype: float64

fix 2, one column per group:
region   Not reported  Outside US     US
quarter                                 
2024Q3            NaN         NaN    NaN
2024Q4         -0.373      -0.027  0.029
2025Q1          0.062      -0.080  0.143


In [ ]:
# 8. Two pandas 3 Habits to Unlearn

In [ ]:
# inplace=True on a column

In [47]:
check = df.merge(reaction_map, on="reaction", how="left")
check["reaction_category"].fillna("Unmapped", inplace=True)
print("blanks after inplace fillna:", check["reaction_category"].isna().sum())

blanks after inplace fillna: 1935


/tmp/ipykernel_2671/2844638142.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  check["reaction_category"].fillna("Unmapped", inplace=True)


In [49]:
check["reaction_category"] = check["reaction_category"].fillna("Unmapped")
print("blanks after assigning back:", check["reaction_category"].isna().sum())

blanks after assigning back: 0


In [ ]:
# 9. Putting It Together: The Lab